# Research: Conventional-Chart Auxiliary SG Loss

This notebook tests the proposed **conventional-chart direct projection auxiliary loss** on a **balanced 20% subset of the train split**, not just a small probe batch.

Main question:

1. Can we reliably recover a primitive-to-conventional lattice transform `C_i` from the current KLDM data frame?
2. Does the space-group projection residual become much smaller in the conventional chart than in the raw primitive/reduced chart?
3. If we softly project in conventional `k`-space and map back, is the induced primitive structure still geometry-safe?

The results are reported both **globally** and **broken down by family**:

- cubic
- hexagonal/trigonal
- tetragonal
- orthorhombic
- monoclinic
- triclinic


## Pass Conditions

The intended success pattern is:

- `conventional residual << primitive residual`
- `fit_max_abs` small
- `oracle match_rate ≈ 1.0` for small `rho`

If that holds broadly on this balanced train subset, then the conventional-chart auxiliary is one of the strongest candidate methods.


In [1]:
from __future__ import annotations

import math
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import yaml
from torch.utils.data import DataLoader, Subset
from IPython.display import display

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
SRC_ROOT = ROOT / 'src'
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from kldmPlus.data import CSPTask, resolve_data_root
from kldmPlus.data.transform import cell_lengths_and_angles
from kldmPlus.symmetry.latticeSymmetry import LatticeSymmetry

try:
    from pymatgen.analysis.structure_matcher import StructureMatcher
    from pymatgen.core import Lattice, Structure
    from pymatgen.symmetry.analyzer import SpacegroupAnalyzer
except ImportError:  # pragma: no cover
    StructureMatcher = Lattice = Structure = SpacegroupAnalyzer = None

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 220)
torch.set_printoptions(precision=6, sci_mode=False)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'ROOT={ROOT}')
print(f'DEVICE={DEVICE}')


MODELS_PROJECT_ROOT: /workspace/.venv/lib/python3.11/site-packages/mattergen
ROOT=/workspace
DEVICE=cpu


In [2]:
CONFIG_PATH = ROOT / 'configs' / 'kldm_plus' / 'mp_20' / 'mp20_lattice_eps_pc.yaml'
DOWNLOAD_DATA = False
SPLIT = 'train'
BATCH_SIZE = 32
SUBSET_FRACTION = 0.1
SUBSET_SEED = 2002
SUBSET_STRATEGY = 'balanced_family'
MAX_GRAPHS = None  # set to an int for a smaller debug sweep after subset selection

SYMPREC = 1e-2
ANGLE_TOL = 5.0
MATCHER_STOL = 0.5
MATCHER_ANGLE_TOL = 10.0
MATCHER_LTOL = 0.3

FIT_MEAN_TOL = 1e-5
FIT_P95_TOL = 1e-4
RHO_VALUES = [1.0]

config = yaml.safe_load(CONFIG_PATH.read_text())
print(f'CONFIG_PATH={CONFIG_PATH}')
print(
    f'split={SPLIT} subset_fraction={SUBSET_FRACTION} subset_seed={SUBSET_SEED} '
    f'subset_strategy={SUBSET_STRATEGY} batch_size={BATCH_SIZE} max_graphs={MAX_GRAPHS}'
)
print(f'rho_values={RHO_VALUES}')


CONFIG_PATH=/workspace/configs/kldm_plus/mp_20/mp20_lattice_eps_pc.yaml
split=train subset_fraction=0.1 subset_seed=2002 subset_strategy=balanced_family batch_size=32 max_graphs=None
rho_values=[1.0]


In [3]:
dataset_cfg = dict(config['dataset'])
model_cfg = dict(config['model'])

task = CSPTask(
    dataset_name=str(dataset_cfg['name']),
    lattice_parameterization=str(model_cfg['lattice_parameterization']),
    lattice_representation=str(dataset_cfg.get('lattice_representation', 'kldm')),
)
root = resolve_data_root(dataset_cfg.get('root'))
dataset_full = task.fit_dataset(root=root, split=SPLIT, download=DOWNLOAD_DATA)

def family_from_sg_local(sg: int) -> str:
    if 195 <= sg <= 230:
        return 'cubic'
    if 143 <= sg <= 194:
        return 'hexagonal_or_trigonal'
    if 75 <= sg <= 142:
        return 'tetragonal'
    if 16 <= sg <= 74:
        return 'orthorhombic'
    if 3 <= sg <= 15:
        return 'monoclinic'
    return 'triclinic'

def make_balanced_subset(dataset, subset_size: int, seed: int, *, group_key):
    generator = torch.Generator().manual_seed(seed)
    grouped_indices = {}
    for idx in range(len(dataset)):
        grouped_indices.setdefault(group_key(dataset[idx]), []).append(idx)

    group_items = sorted(grouped_indices.items(), key=lambda item: str(item[0]))
    shuffled_groups = []
    for group, indices in group_items:
        order = torch.randperm(len(indices), generator=generator).tolist()
        shuffled_groups.append((group, [indices[pos] for pos in order]))

    selected = []
    while len(selected) < int(subset_size):
        made_progress = False
        for _group, indices in shuffled_groups:
            if not indices:
                continue
            selected.append(indices.pop(0))
            made_progress = True
            if len(selected) >= int(subset_size):
                break
        if not made_progress:
            break

    return Subset(dataset, selected)

if SUBSET_FRACTION is None or float(SUBSET_FRACTION) <= 0.0 or float(SUBSET_FRACTION) >= 1.0:
    dataset = dataset_full
else:
    subset_size = max(1, int(round(len(dataset_full) * float(SUBSET_FRACTION))))
    if SUBSET_STRATEGY == 'balanced_family':
        dataset = make_balanced_subset(
            dataset_full,
            subset_size=subset_size,
            seed=SUBSET_SEED,
            group_key=lambda sample: family_from_sg_local(int(torch.as_tensor(sample.space_group).reshape(-1)[0].item())),
        )
    elif SUBSET_STRATEGY == 'balanced_space_group':
        dataset = make_balanced_subset(
            dataset_full,
            subset_size=subset_size,
            seed=SUBSET_SEED,
            group_key=lambda sample: int(torch.as_tensor(sample.space_group).reshape(-1)[0].item()),
        )
    else:
        generator = torch.Generator().manual_seed(SUBSET_SEED)
        indices = torch.randperm(len(dataset_full), generator=generator)[:subset_size].tolist()
        dataset = Subset(dataset_full, indices)

loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=False,
    collate_fn=dataset_full.collate_fn,
)
lattice_transform = task.make_lattice_transform(root=root, download=DOWNLOAD_DATA)
lattice_symmetry = LatticeSymmetry().to(DEVICE)

print(f'dataset_full_size={len(dataset_full)}')
print(f'dataset_subset_size={len(dataset)}')
print(f'lattice_representation={task.lattice_representation} lattice_parameterization={model_cfg["lattice_parameterization"]}')


dataset_cache action=load dataset=mp_20 split=train reason=fresh path=/workspace/data/mp_20/processed/train
dataset_cache action=from_cache_path:start dataset=mp_20 split=train
dataset_cache action=from_cache_path:done dataset=mp_20 split=train
dataset_cache action=builder_build:start dataset=mp_20 split=train
dataset_cache action=builder_build:done dataset=mp_20 split=train
dataset_full_size=27136
dataset_subset_size=2714
lattice_representation=kldm lattice_parameterization=eps


## Helpers

In [4]:
def safe_float(value):
    try:
        return float(value)
    except Exception:
        return float('nan')


def display_sorted_frame(rows_or_frame, sort_by=None):
    frame = rows_or_frame if isinstance(rows_or_frame, pd.DataFrame) else pd.DataFrame(rows_or_frame)
    if sort_by:
        frame = frame.sort_values(sort_by).reset_index(drop=True)
    display(frame)
    return frame


def family_from_sg(sg: int) -> str:
    if 195 <= sg <= 230:
        return 'cubic'
    if 143 <= sg <= 194:
        return 'hexagonal_or_trigonal'
    if 75 <= sg <= 142:
        return 'tetragonal'
    if 16 <= sg <= 74:
        return 'orthorhombic'
    if 3 <= sg <= 15:
        return 'monoclinic'
    return 'triclinic'


def space_group_tensor(batch) -> torch.Tensor:
    for name in ('space_group', 'spacegroup'):
        if hasattr(batch, name):
            return torch.as_tensor(getattr(batch, name), device=batch.pos.device, dtype=torch.long).reshape(-1)
    raise AttributeError('Batch is missing space_group / spacegroup.')


def graph_masks(batch):
    return [batch.batch == g for g in range(int(batch.num_graphs))]


def lengths_angles_to_matrix(lengths: torch.Tensor, angles: torch.Tensor) -> torch.Tensor:
    a, b, c = lengths.unbind(dim=-1)
    alpha, beta, gamma = angles.unbind(dim=-1)
    cos_alpha = torch.cos(alpha)
    cos_beta = torch.cos(beta)
    cos_gamma = torch.cos(gamma)
    sin_gamma = torch.sin(gamma).clamp_min(1e-8)
    row0 = torch.stack([a, torch.zeros_like(a), torch.zeros_like(a)], dim=-1)
    row1 = torch.stack([b * cos_gamma, b * sin_gamma, torch.zeros_like(b)], dim=-1)
    cx = c * cos_beta
    cy = c * (cos_alpha - cos_beta * cos_gamma) / sin_gamma
    cz_sq = (c.square() - cx.square() - cy.square()).clamp_min(1e-12)
    row2 = torch.stack([cx, cy, torch.sqrt(cz_sq)], dim=-1)
    return torch.stack([row0, row1, row2], dim=-2)


def lattice6_to_matrix(l: torch.Tensor, num_atoms: torch.Tensor | int, lattice_transform) -> torch.Tensor:
    if hasattr(lattice_transform, 'invert_to_matrix'):
        return lattice_transform.invert_to_matrix(l=l, num_atoms=num_atoms).reshape(*l.shape[:-1], 3, 3)
    lengths, angles = lattice_transform.invert_to_lengths_angles(l=l, num_atoms=num_atoms)
    return lengths_angles_to_matrix(lengths, angles)


def source_cells_from_batch(batch, lattice_transform) -> torch.Tensor:
    if hasattr(batch, 'cell'):
        cell = torch.as_tensor(batch.cell, device=batch.pos.device, dtype=batch.pos.dtype)
        if cell.ndim == 4 and cell.shape[1] == 1:
            return cell[:, 0]
        if cell.ndim == 3:
            return cell
    return lattice6_to_matrix(batch.l, batch.num_atoms, lattice_transform)


def build_structure(cell: torch.Tensor, frac: torch.Tensor, atomic_numbers: torch.Tensor):
    if None in (Lattice, Structure):
        return None
    lattice = Lattice(cell.detach().cpu().numpy())
    species = [int(z) for z in atomic_numbers.detach().cpu().tolist()]
    coords = frac.detach().cpu().numpy()
    return Structure(lattice, species, coords, coords_are_cartesian=False)


def conventional_standard_structure(structure):
    if structure is None or SpacegroupAnalyzer is None:
        return None
    analyzer = SpacegroupAnalyzer(structure, symprec=SYMPREC, angle_tolerance=ANGLE_TOL)
    return analyzer.get_conventional_standard_structure().get_sorted_structure()


def detect_space_group(structure):
    if structure is None or SpacegroupAnalyzer is None:
        return None
    try:
        return int(SpacegroupAnalyzer(structure, symprec=SYMPREC, angle_tolerance=ANGLE_TOL).get_space_group_number())
    except Exception:
        return None


def matcher_fit_and_rms(source, target):
    if source is None or target is None or StructureMatcher is None:
        return None, None
    matcher = StructureMatcher(stol=MATCHER_STOL, angle_tol=MATCHER_ANGLE_TOL, ltol=MATCHER_LTOL)
    fit = bool(matcher.fit(source, target))
    try:
        rms = matcher.get_rms_dist(source, target)
        rms_value = None if rms is None else float(rms[0] if isinstance(rms, tuple) else rms)
    except Exception:
        rms_value = None
    return fit, rms_value


def pairwise_pbc_distances(frac: torch.Tensor, cell: torch.Tensor) -> torch.Tensor:
    delta = frac[:, None, :] - frac[None, :, :]
    delta = delta - torch.round(delta)
    cart = torch.einsum('...i,ij->...j', delta, cell)
    return torch.linalg.norm(cart, dim=-1)


def lattice_to_k(cell: torch.Tensor) -> torch.Tensor:
    sym = lattice_symmetry.de_so3(cell[None])[0]
    return lattice_symmetry.m2v(sym[None])[0]


def k_projection_residual(cell: torch.Tensor, sg: int) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    k = lattice_to_k(cell)
    sg_tensor = torch.tensor([int(sg)], device=cell.device, dtype=torch.long)
    k_proj = lattice_symmetry.proj_k_to_spacegroup(k[None], sg_tensor)[0]
    residual = torch.abs(k - k_proj)
    return k, k_proj, residual


def cell_lengths_angles(cell: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    return cell_lengths_and_angles(cell)


def lattice_error_metrics(cell_ref: torch.Tensor, cell_new: torch.Tensor) -> dict[str, float]:
    len_ref, ang_ref = cell_lengths_angles(cell_ref)
    len_new, ang_new = cell_lengths_angles(cell_new)
    lengths_rmse = torch.sqrt(torch.mean((len_new - len_ref).square()))
    angles_rmse = torch.sqrt(torch.mean(((ang_new - ang_ref) * 180.0 / math.pi).square()))
    vol_ref = torch.det(cell_ref).abs().clamp_min(1e-12)
    vol_new = torch.det(cell_new).abs()
    volume_rel_error = torch.abs(vol_new - vol_ref) / vol_ref
    return {
        'lengths_rmse': safe_float(lengths_rmse),
        'angles_rmse_deg': safe_float(angles_rmse),
        'volume_rel_error': safe_float(volume_rel_error),
    }


def induced_cart_shift_metrics(frac: torch.Tensor, cell_ref: torch.Tensor, cell_new: torch.Tensor) -> dict[str, float]:
    delta = frac @ (cell_new - cell_ref)
    norms = torch.linalg.norm(delta, dim=1)
    return {
        'induced_cart_shift_rms': safe_float(torch.sqrt(torch.mean(norms.square()))),
        'induced_cart_shift_max': safe_float(torch.max(norms)),
    }


def family_summary(frame: pd.DataFrame, *, group_cols=('family',), rho=None):
    work = frame.copy()
    if rho is not None and 'rho' in work.columns:
        work = work[work['rho'] == rho].copy()
    if len(work) == 0:
        return pd.DataFrame()
    agg = {
        'graph_id': ('graph_id', 'count'),
    }
    if 'fit_max_abs' in work.columns:
        agg.update({
            'fit_mean': ('fit_max_abs', 'mean'),
            'fit_p95': ('fit_max_abs', lambda s: float(pd.Series(s).quantile(0.95))),
            'fit_pass_rate': ('PASS_fit', 'mean'),
        })
    if 'primitive_proj_abs_mean' in work.columns:
        agg.update({
            'primitive_proj_mean': ('primitive_proj_abs_mean', 'mean'),
            'conventional_proj_mean': ('conventional_proj_abs_mean', 'mean'),
            'improvement_abs_mean': ('improvement_abs', 'mean'),
            'improvement_ratio_mean': ('improvement_ratio', 'mean'),
            'conv_better_rate': ('PASS_conventional_better', 'mean'),
        })
    if 'structure_match' in work.columns:
        agg.update({
            'match_rate': ('structure_match', lambda s: float(pd.Series(s).fillna(False).mean())),
            'lengths_rmse_mean': ('lengths_rmse', 'mean'),
            'angles_rmse_mean_deg': ('angles_rmse_deg', 'mean'),
            'volume_rel_error_mean': ('volume_rel_error', 'mean'),
            'induced_cart_shift_rms_mean': ('induced_cart_shift_rms', 'mean'),
            'induced_cart_shift_rms_p95': ('induced_cart_shift_rms', lambda s: float(pd.Series(s).quantile(0.95))),
        })
    return work.groupby(list(group_cols), dropna=False).agg(**agg).reset_index()


## Full-Train Diagnostic Sweep

This scans a balanced 20% subset of the train split and records per-graph rows for all three tests.


In [5]:
rows_t1 = []
rows_t2 = []
rows_t3 = []
graph_counter = 0

for batch_idx, batch in enumerate(loader):
    batch = batch.to(DEVICE)
    cells = source_cells_from_batch(batch, lattice_transform)
    sg = space_group_tensor(batch)

    for local_graph, node_mask in enumerate(graph_masks(batch)):
        if MAX_GRAPHS is not None and graph_counter >= int(MAX_GRAPHS):
            break

        graph_id = int(graph_counter)
        graph_counter += 1

        cell_p = cells[local_graph].reshape(3, 3)
        frac = batch.pos[node_mask].reshape(-1, 3)
        atomic_numbers = batch.atomic_numbers[node_mask].reshape(-1)
        sg_i = int(sg[local_graph].item())
        family = family_from_sg(sg_i)

        structure_p = build_structure(cell_p, frac, atomic_numbers)
        structure_c = conventional_standard_structure(structure_p)

        base_row = {
            'graph_id': graph_id,
            'batch_idx': batch_idx,
            'local_graph': int(local_graph),
            'space_group': sg_i,
            'family': family,
            'atom_count_raw': int(frac.shape[0]),
        }

        if structure_c is None:
            rows_t1.append({**base_row, 'status': 'missing_conventional_structure', 'fit_max_abs': float('nan'), 'fit_fro': float('nan'), 'atom_count_conv': None, 'atom_count_delta': None, 'PASS_fit': False})
            rows_t2.append({**base_row, 'primitive_proj_abs_mean': float('nan'), 'primitive_proj_abs_max': float('nan'), 'conventional_proj_abs_mean': float('nan'), 'conventional_proj_abs_max': float('nan'), 'improvement_abs': float('nan'), 'improvement_ratio': float('nan'), 'PASS_conventional_better': False})
            for rho in RHO_VALUES:
                rows_t3.append({**base_row, 'rho': float(rho), 'structure_match': None, 'structure_rms': None, 'detected_sg_proj': None, 'lengths_rmse': float('nan'), 'angles_rmse_deg': float('nan'), 'volume_rel_error': float('nan'), 'induced_cart_shift_rms': float('nan'), 'induced_cart_shift_max': float('nan')})
            continue

        cell_c = torch.tensor(np.asarray(structure_c.lattice.matrix), device=cell_p.device, dtype=cell_p.dtype)
        c_i = cell_c @ torch.linalg.inv(cell_p)
        c_i_inv = torch.linalg.inv(c_i)
        fit = c_i @ cell_p - cell_c

        rows_t1.append({
            **base_row,
            'status': 'ok',
            'atom_count_conv': int(len(structure_c)),
            'atom_count_delta': int(len(structure_c) - int(frac.shape[0])),
            'det_primitive': safe_float(torch.det(cell_p)),
            'det_conventional': safe_float(torch.det(cell_c)),
            'fit_max_abs': safe_float(torch.max(torch.abs(fit))),
            'fit_fro': safe_float(torch.linalg.norm(fit.reshape(-1))),
            'PASS_fit': bool(torch.max(torch.abs(fit)) < FIT_P95_TOL),
        })

        _, _, residual_p = k_projection_residual(cell_p, sg_i)
        k_c, k_proj, residual_c = k_projection_residual(cell_c, sg_i)
        mean_p = torch.mean(residual_p)
        mean_c = torch.mean(residual_c)
        rows_t2.append({
            **base_row,
            'primitive_proj_abs_mean': safe_float(mean_p),
            'primitive_proj_abs_max': safe_float(torch.max(residual_p)),
            'conventional_proj_abs_mean': safe_float(mean_c),
            'conventional_proj_abs_max': safe_float(torch.max(residual_c)),
            'improvement_abs': safe_float(mean_p - mean_c),
            'improvement_ratio': safe_float(mean_c / mean_p.clamp_min(1e-12)),
            'PASS_conventional_better': bool(mean_c < mean_p),
        })

        for rho in RHO_VALUES:
            k_rho = (1.0 - rho) * k_c + rho * k_proj
            cell_c_rho = lattice_symmetry.v2m(k_rho[None])[0]
            cell_p_rho = c_i_inv @ cell_c_rho
            structure_p_rho = build_structure(cell_p_rho, frac, atomic_numbers)
            fit_match, rms = matcher_fit_and_rms(structure_p, structure_p_rho)
            metrics = lattice_error_metrics(cell_p, cell_p_rho)
            shift = induced_cart_shift_metrics(frac, cell_p, cell_p_rho)
            rows_t3.append({
                **base_row,
                'rho': float(rho),
                'conventional_proj_abs_mean': safe_float(torch.mean(residual_c)),
                'structure_match': fit_match,
                'structure_rms': rms,
                'detected_sg_proj': detect_space_group(structure_p_rho),
                **metrics,
                **shift,
            })

    if MAX_GRAPHS is not None and graph_counter >= int(MAX_GRAPHS):
        break

print(f'processed_graphs={graph_counter}')
df_t1 = pd.DataFrame(rows_t1)
df_t2 = pd.DataFrame(rows_t2)
df_t3 = pd.DataFrame(rows_t3)
print(df_t1.shape, df_t2.shape, df_t3.shape)


processed_graphs=2714
(2714, 14) (2714, 13) (2714, 16)


## Test 1 Summary: Conventional Transform Recovery

In [6]:
df_t1_global = pd.DataFrame([{
    'n': int(len(df_t1)),
    'fit_mean': safe_float(df_t1['fit_max_abs'].mean()),
    'fit_p95': safe_float(df_t1['fit_max_abs'].quantile(0.95)),
    'fit_pass_rate': safe_float(df_t1['PASS_fit'].mean()),
    'atom_count_delta_nonzero_rate': safe_float((df_t1['atom_count_delta'].fillna(0) != 0).mean()),
}])
display(df_t1_global)
df_t1_family = family_summary(df_t1)
display(df_t1_family.sort_values('family'))


,n,fit_mean,fit_p95,fit_pass_rate,atom_count_delta_nonzero_rate
0,2714,4.601818e-07,9.536743e-07,1.0,0.507001


,family,graph_id,fit_mean,fit_p95,fit_pass_rate
0,cubic,453,4.995751e-07,9.536743e-07,1.0
1,hexagonal_or_trigonal,453,4.955106e-07,1.907349e-06,1.0
2,monoclinic,452,4.859578e-07,9.536743e-07,1.0
3,orthorhombic,452,4.353431e-07,9.536743e-07,1.0
4,tetragonal,452,4.573530e-07,9.961477e-07,1.0
5,triclinic,452,3.871862e-07,9.536743e-07,1.0


## Test 2 Summary: Primitive vs Conventional Residuals

In [7]:
df_t2_global = pd.DataFrame([{
    'n': int(len(df_t2)),
    'primitive_proj_mean': safe_float(df_t2['primitive_proj_abs_mean'].mean()),
    'conventional_proj_mean': safe_float(df_t2['conventional_proj_abs_mean'].mean()),
    'improvement_abs_mean': safe_float(df_t2['improvement_abs'].mean()),
    'improvement_ratio_mean': safe_float(df_t2['improvement_ratio'].mean()),
    'conv_better_rate': safe_float(df_t2['PASS_conventional_better'].mean()),
}])
display(df_t2_global)
df_t2_family = family_summary(df_t2)
display(df_t2_family.sort_values('family'))


,n,primitive_proj_mean,conventional_proj_mean,improvement_abs_mean,improvement_ratio_mean,conv_better_rate
0,2714,0.068833,0.001175,0.067658,171.442949,0.682756


,family,graph_id,primitive_proj_mean,conventional_proj_mean,improvement_abs_mean,improvement_ratio_mean,conv_better_rate
0,cubic,453,0.134316,0.000112,0.134204,0.247643,0.843267
1,hexagonal_or_trigonal,453,0.111534,0.004256,0.107278,1.324272,0.885210
2,monoclinic,452,0.045342,0.001682,0.043659,586.177765,0.922566
3,orthorhombic,452,0.043376,0.000714,0.042662,441.276727,0.681416
4,tetragonal,452,0.078191,0.000280,0.077911,0.386408,0.763274
5,triclinic,452,0.000000,0.000000,0.000000,0.000000,0.000000


## Test 3 Summary: Oracle Safety After Mapping Back

The family tables below are the most important ones for the soft auxiliary-loss interpretation.


In [8]:
df_t3_global = family_summary(df_t3, group_cols=('rho',))
display(df_t3_global.sort_values('rho'))

small_rho_family = family_summary(df_t3, rho=0.05)
if len(small_rho_family):
    display(small_rho_family.sort_values('family'))
else:
    display(pd.DataFrame(columns=['family', 'graph_id', 'match_rate', 'lengths_rmse_mean', 'angles_rmse_mean_deg', 'volume_rel_error_mean', 'induced_cart_shift_rms_mean', 'induced_cart_shift_rms_p95']))

small_rho_family_008 = family_summary(df_t3, rho=0.08)
if len(small_rho_family_008):
    display(small_rho_family_008.sort_values('family'))
else:
    display(pd.DataFrame(columns=['family', 'graph_id', 'match_rate', 'lengths_rmse_mean', 'angles_rmse_mean_deg', 'volume_rel_error_mean', 'induced_cart_shift_rms_mean', 'induced_cart_shift_rms_p95']))

hard_rho_family = family_summary(df_t3, rho=1.0)
if len(hard_rho_family):
    display(hard_rho_family.sort_values('family'))
else:
    display(pd.DataFrame(columns=['family', 'graph_id', 'match_rate', 'lengths_rmse_mean', 'angles_rmse_mean_deg', 'volume_rel_error_mean', 'induced_cart_shift_rms_mean', 'induced_cart_shift_rms_p95']))


,rho,graph_id,match_rate,lengths_rmse_mean,angles_rmse_mean_deg,volume_rel_error_mean,induced_cart_shift_rms_mean,induced_cart_shift_rms_p95
0,1.0,2714,0.989683,0.011866,0.15322,5.368292e-07,2.549031,11.85115


,family,graph_id,match_rate,lengths_rmse_mean,angles_rmse_mean_deg,volume_rel_error_mean,induced_cart_shift_rms_mean,induced_cart_shift_rms_p95


,family,graph_id,match_rate,lengths_rmse_mean,angles_rmse_mean_deg,volume_rel_error_mean,induced_cart_shift_rms_mean,induced_cart_shift_rms_p95


,family,graph_id,match_rate,lengths_rmse_mean,angles_rmse_mean_deg,volume_rel_error_mean,induced_cart_shift_rms_mean,induced_cart_shift_rms_p95
0,cubic,453,1.000000,0.001888,0.010909,4.351210e-07,0.001121,0.000003
1,hexagonal_or_trigonal,453,0.969095,0.037293,0.525969,4.418271e-07,2.761799,4.566857
2,monoclinic,452,0.980088,0.016085,0.250692,5.826029e-07,9.206282,15.326572
3,orthorhombic,452,0.991150,0.009957,0.106789,4.460758e-07,0.100180,0.000003
4,tetragonal,452,0.997788,0.005933,0.024431,5.001359e-07,0.034075,0.000003
5,triclinic,452,1.000000,0.000003,0.000020,8.156476e-07,3.195895,12.639029


## Combined Family Verdict Table

This combines the three pass conditions family-by-family:

- small transform fit
- conventional residual better than primitive residual
- oracle match rate near 1.0 for `rho=0.05`


In [9]:
fit_cols = df_t1_family[['family', 'fit_mean', 'fit_p95', 'fit_pass_rate']].copy()
res_cols = df_t2_family[['family', 'primitive_proj_mean', 'conventional_proj_mean', 'improvement_abs_mean', 'conv_better_rate']].copy()
rho005_cols = (
    small_rho_family[['family', 'match_rate', 'angles_rmse_mean_deg', 'volume_rel_error_mean', 'induced_cart_shift_rms_mean']].copy()
    if len(small_rho_family)
    else pd.DataFrame(columns=['family', 'match_rate', 'angles_rmse_mean_deg', 'volume_rel_error_mean', 'induced_cart_shift_rms_mean'])
)
combined = fit_cols.merge(res_cols, on='family', how='outer').merge(rho005_cols, on='family', how='outer')
combined['PASS_fit_small'] = combined['fit_p95'] < FIT_P95_TOL
combined['PASS_conv_much_better'] = combined['conventional_proj_mean'] < combined['primitive_proj_mean']
combined['PASS_rho005_match'] = combined['match_rate'] >= 0.95
combined = combined.sort_values('family').reset_index(drop=True)
display(combined)


,family,fit_mean,fit_p95,fit_pass_rate,primitive_proj_mean,conventional_proj_mean,improvement_abs_mean,conv_better_rate,match_rate,angles_rmse_mean_deg,volume_rel_error_mean,induced_cart_shift_rms_mean,PASS_fit_small,PASS_conv_much_better,PASS_rho005_match
0,cubic,4.995751e-07,9.536743e-07,1.0,0.134316,0.000112,0.134204,0.843267,NaN,NaN,NaN,NaN,True,True,False
1,hexagonal_or_trigonal,4.955106e-07,1.907349e-06,1.0,0.111534,0.004256,0.107278,0.885210,NaN,NaN,NaN,NaN,True,True,False
2,monoclinic,4.859578e-07,9.536743e-07,1.0,0.045342,0.001682,0.043659,0.922566,NaN,NaN,NaN,NaN,True,True,False
3,orthorhombic,4.353431e-07,9.536743e-07,1.0,0.043376,0.000714,0.042662,0.681416,NaN,NaN,NaN,NaN,True,True,False
4,tetragonal,4.573530e-07,9.961477e-07,1.0,0.078191,0.000280,0.077911,0.763274,NaN,NaN,NaN,NaN,True,True,False
5,triclinic,3.871862e-07,9.536743e-07,1.0,0.000000,0.000000,0.000000,0.000000,NaN,NaN,NaN,NaN,True,False,False


## Final Verdict

Interpretation guide:

- If `fit_p95` is tiny, the conventional transform is numerically well-defined.
- If `conventional_proj_mean << primitive_proj_mean`, the auxiliary is correcting the right chart.
- If `rho=0.05` or `0.08` keeps `match_rate ≈ 1.0`, then the soft conventional auxiliary is likely safe even if `rho=1.0` is too strong.


In [10]:
global_fit_mean = safe_float(df_t1['fit_max_abs'].mean()) if len(df_t1) else float('nan')
global_fit_p95 = safe_float(df_t1['fit_max_abs'].quantile(0.95)) if len(df_t1) else float('nan')
global_dp = safe_float(df_t2['primitive_proj_abs_mean'].mean()) if len(df_t2) else float('nan')
global_dc = safe_float(df_t2['conventional_proj_abs_mean'].mean()) if len(df_t2) else float('nan')
small_rho_global = df_t3_global[df_t3_global['rho'] == 0.05]
small_match = safe_float(small_rho_global['match_rate'].iloc[0]) if len(small_rho_global) else float('nan')

print('Global transform recovery:')
print(f'  mean fit_max_abs = {global_fit_mean:.6g}')
print(f'  p95  fit_max_abs = {global_fit_p95:.6g}')
print('Global residual comparison:')
print(f'  primitive mean residual     = {global_dp:.6g}')
print(f'  conventional mean residual = {global_dc:.6g}')
print('Global oracle safety at rho=0.05:')
print(f'  match_rate = {small_match:.6g}')

if global_dc < global_dp:
    print('Notebook verdict: the conventional chart looks better than the primitive chart globally; check the family tables to decide whether the auxiliary is broadly safe enough.')
else:
    print('Notebook verdict: the conventional chart is not globally better than the primitive chart, so this auxiliary may not be worth adding.')


Global transform recovery:
  mean fit_max_abs = 4.60182e-07
  p95  fit_max_abs = 9.53674e-07
Global residual comparison:
  primitive mean residual     = 0.0688331
  conventional mean residual = 0.0011749
Global oracle safety at rho=0.05:
  match_rate = nan
Notebook verdict: the conventional chart looks better than the primitive chart globally; check the family tables to decide whether the auxiliary is broadly safe enough.
